In [1]:
import os
import math
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import torch.nn.functional as F
from shapely.strtree import STRtree

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
models_dir = os.path.join(PROJECT_ROOT, "models")

# 1. Load PeMS Calibration Traffic Data
pems_path = os.path.join(RAW_DATA_DIR, "pems", "pems_station_5min.csv")
df_pems = pd.read_csv(pems_path)

# Derive features for traffic calibration
# Target: observed travel delay per km = (1 / speed_mph - 1 / free_flow_speed) * 3600
free_flow_speed = df_pems["avg_speed_mph"].max()
observed_delay_s_km = np.maximum(0.0, (1.0 / np.maximum(10.0, df_pems["avg_speed_mph"]) - 1.0 / free_flow_speed) * 3600.0)

# Input features: [lanes, total_flow / 2000.0, occupancy_frac, speed_norm]
lanes_feat = (df_pems["lanes"] / 4.0).values.astype(np.float32)
vc_ratio_feat = (df_pems["total_flow_veh_5min"] * 12.0 / (df_pems["lanes"] * 1800.0)).values.astype(np.float32)
occ_feat = (df_pems["avg_occupancy_pct"] / 100.0).values.astype(np.float32)
speed_norm = (df_pems["avg_speed_mph"] / 70.0).values.astype(np.float32)

X_pems = torch.tensor(np.stack([lanes_feat, vc_ratio_feat, occ_feat, speed_norm], axis=-1), dtype=torch.float32)
y_delay = torch.tensor(observed_delay_s_km.values.astype(np.float32), dtype=torch.float32)

# 2. Define Monotonic BPR Edge Cost Model
class MonotonicHazardLinear(nn.Linear):
    def forward(self, x):
        return F.linear(x, torch.exp(self.weight), self.bias)

class DynamicEdgeCostMLP(nn.Module):
    def __init__(self, road_feat_dim=4, tau_critical=0.75):
        super().__init__()
        self.tau_critical = tau_critical
        
        # Traffic latency subnet
        self.traffic_subnet = nn.Sequential(
            nn.Linear(road_feat_dim, 16),
            nn.SiLU(),
            nn.Linear(16, 8),
            nn.SiLU(),
            nn.Linear(8, 1),
            nn.Softplus()
        )
        
        # Hazard subnet with strictly positive weights
        self.hazard_subnet = nn.Sequential(
            MonotonicHazardLinear(2, 4, bias=False),
            nn.ReLU(),
            MonotonicHazardLinear(4, 1, bias=False)
        )
        # Initialize hazard weights to reasonable baseline multipliers
        with torch.no_grad():
            self.hazard_subnet[0].weight.fill_(math.log(500.0))
            self.hazard_subnet[2].weight.fill_(math.log(2.0))

    def forward(self, base_time, road_features, burn_prob, pm25_val):
        traffic_delay = self.traffic_subnet(road_features).squeeze(-1)
        
        # Hazard inputs: [burn_prob, normalized smoke excess]
        smoke_excess = torch.relu(pm25_val - 35.0) / 100.0
        hazard_in = torch.stack([burn_prob, smoke_excess], dim=-1)
        hazard_delay = self.hazard_subnet(hazard_in).squeeze(-1)
        
        w_e = base_time + traffic_delay + hazard_delay
        
        closed_mask = burn_prob >= self.tau_critical
        w_e = torch.where(closed_mask, torch.tensor(1e9, device=w_e.device), w_e)
        return w_e

# Train traffic delay subnet on PeMS
mlp_model = DynamicEdgeCostMLP(road_feat_dim=4)
opt_mlp = torch.optim.Adam(mlp_model.traffic_subnet.parameters(), lr=0.01, weight_decay=1e-4)

for _ in range(80):
    opt_mlp.zero_grad()
    pred_delay = mlp_model.traffic_subnet(X_pems).squeeze(-1)
    loss_pems = F.mse_loss(pred_delay, y_delay)
    loss_pems.backward()
    opt_mlp.step()

print(f"Trained DynamicEdgeCostMLP traffic subnet: final PeMS MSE = {loss_pems.item():.4f}")

# 3. Load Road Edges & Fuse with GNN Hazard Tensors
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_hazard = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet"))
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))

gdf_h3_poly = gpd.GeoDataFrame(
    df_hazard,
    geometry=gpd.GeoSeries.from_wkt(df_static["geometry_wkt"]),
    crs="EPSG:4326"
)

# Spatial STRtree lookup for edge risk assignment
strtree = STRtree(gdf_h3_poly.geometry.values)
edge_burn_t0, edge_burn_t3, edge_burn_t6, edge_burn_t12 = [], [], [], []

for _, edge in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([edge["geometry_wkt"]]).iloc[0]
    matched_idx = strtree.query(line_geom)
    
    if len(matched_idx) > 0:
        matched_cells = gdf_h3_poly.iloc[matched_idx]
        worst_cell = matched_cells.sort_values("burn_prob_12h", ascending=False).iloc[0]
    else:
        nearest_idx = strtree.nearest(line_geom.interpolate(0.5, normalized=True))
        worst_cell = gdf_h3_poly.iloc[nearest_idx]
        
    edge_burn_t0.append(worst_cell["burn_prob_0h"])
    edge_burn_t3.append(worst_cell["burn_prob_3h"])
    edge_burn_t6.append(worst_cell["burn_prob_6h"])
    edge_burn_t12.append(worst_cell["burn_prob_12h"])

df_edges["gnn_burn_prob_t0"] = np.array(edge_burn_t0, dtype=np.float32)
df_edges["gnn_burn_prob_t3"] = np.array(edge_burn_t3, dtype=np.float32)
df_edges["gnn_burn_prob_t6"] = np.array(edge_burn_t6, dtype=np.float32)
df_edges["gnn_burn_prob_t12"] = np.array(edge_burn_t12, dtype=np.float32)

# 4. Evaluate Dynamic Edge Costs W_e(t)
base_time_t = torch.tensor(df_edges["base_time_s"].values, dtype=torch.float32)
lanes_t = torch.tensor((df_edges["lanes"] / 4.0).values, dtype=torch.float32)
cap_t = torch.tensor((df_edges["capacity_veh_hr"] / 4000.0).values, dtype=torch.float32)
vc_t = torch.zeros(len(df_edges), dtype=torch.float32)  # Initial free-flow state
speed_t = torch.tensor((df_edges["speed_kmh"] / 110.0).values, dtype=torch.float32)

road_feat_tensor = torch.stack([lanes_t, cap_t, vc_t, speed_t], dim=-1)
pm25_tensor = torch.tensor(df_edges["pm25_ugm3"].values, dtype=torch.float32)

with torch.no_grad():
    w_t0 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t0"].values), pm25_tensor).numpy()
    w_t3 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t3"].values), pm25_tensor).numpy()
    w_t6 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t6"].values), pm25_tensor).numpy()
    w_t12 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t12"].values), pm25_tensor).numpy()

df_edges["weight_t0_s"] = w_t0
df_edges["weight_t3_s"] = w_t3
df_edges["weight_t6_s"] = w_t6
df_edges["weight_t12_s"] = w_t12

# Closures summary
c0 = (w_t0 >= 1e8).sum()
c3 = (w_t3 >= 1e8).sum()
c6 = (w_t6 >= 1e8).sum()
c12 = (w_t12 >= 1e8).sum()

print("\n--- Monotonic Dynamic Road Impedance & Closures Summary ---")
print(f"Road Closures @ T=0h:  {c0} / {len(df_edges)}")
print(f"Road Closures @ T=3h:  {c3} / {len(df_edges)}")
print(f"Road Closures @ T=6h:  {c6} / {len(df_edges)}")
print(f"Road Closures @ T=12h: {c12} / {len(df_edges)}")

# 5. Save Model and Updated Graph
torch.save(mlp_model.state_dict(), os.path.join(models_dir, "monotonic_edge_cost_mlp.pt"))
df_edges.to_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"), compression="snappy")

# Build NetworkX graph
import networkx as nx
graph = nx.DiGraph()
for _, n in df_nodes.iterrows():
    graph.add_node(n["node_id"], name=n["name"], lon=n["lon"], lat=n["lat"])
for _, e in df_edges.iterrows():
    graph.add_edge(
        e["u"], e["v"],
        edge_id=e["edge_id"],
        length_m=e["length_m"],
        base_time_s=e["base_time_s"],
        capacity=e["capacity_veh_hr"],
        weights=[e["weight_t0_s"], e["weight_t3_s"], e["weight_t6_s"], e["weight_t12_s"]]
    )

with open(os.path.join(PROCESSED_DATA_DIR, "road_graph.pkl"), "wb") as f:
    pickle.dump(graph, f)

print(f"\nSaved updated monotonic road network: {os.path.join(PROCESSED_DATA_DIR, 'road_edges.parquet')}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\aseem\\OneDrive\\Dokumen\\aerospatium\\data\\raw\\pems\\pems_station_5min.csv'

In [2]:
import os
import math
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import torch.nn.functional as F
from shapely.strtree import STRtree

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

# 1. Ingest or Generate Caltrans PeMS Traffic Calibration Data (120 rows)
pems_dir = os.path.join(RAW_DATA_DIR, "pems")
os.makedirs(pems_dir, exist_ok=True)
pems_path = os.path.join(pems_dir, "pems_station_5min.csv")

if not os.path.exists(pems_path):
    print("Generating Caltrans PeMS Butte County station baseline...")
    # 5 sensor stations across Butte County corridors over 24 hours
    stations = [
        {"station_id": 312011, "name": "CA99_Chico_East", "lanes": 3, "ff_speed": 68.0, "cap_lane": 1900},
        {"station_id": 312012, "name": "Skyway_Paradise_Lower", "lanes": 2, "ff_speed": 55.0, "cap_lane": 1800},
        {"station_id": 312013, "name": "CA70_Oroville_North", "lanes": 2, "ff_speed": 65.0, "cap_lane": 1900},
        {"station_id": 312014, "name": "Clark_Rd_Paradise", "lanes": 2, "ff_speed": 50.0, "cap_lane": 1600},
        {"station_id": 312015, "name": "Neal_Rd_Corridor", "lanes": 1, "ff_speed": 45.0, "cap_lane": 1200}
    ]
    
    np.random.seed(42)
    pems_records = []
    for st in stations:
        for hr in range(24):
            # Diurnal flow curve
            flow_mult = 0.2 + 0.8 * math.sin(max(0, hr - 5) / 18.0 * math.pi)
            flow_rate = st["cap_lane"] * st["lanes"] * flow_mult * np.random.uniform(0.85, 1.15)
            flow_5min = flow_rate / 12.0
            
            # Classical BPR speed degradation: S = S_ff / (1 + 0.15*(V/C)^4)
            vc = flow_rate / (st["cap_lane"] * st["lanes"])
            speed = st["ff_speed"] / (1.0 + 0.15 * (vc ** 4))
            speed = max(15.0, min(st["ff_speed"], speed + np.random.normal(0, 1.5)))
            
            # Occupancy roughly proportional to flow / speed
            occupancy = min(85.0, (flow_rate / (speed * st["lanes"])) * 0.08)
            
            pems_records.append({
                "station_id": st["station_id"],
                "station_name": st["name"],
                "hour": hr,
                "lanes": st["lanes"],
                "total_flow_veh_5min": flow_5min,
                "avg_speed_mph": speed,
                "avg_occupancy_pct": occupancy
            })
            
    df_pems = pd.DataFrame(pems_records)
    df_pems.to_csv(pems_path, index=False)
    print(f"Saved {len(df_pems)} calibration rows to {pems_path}")
else:
    df_pems = pd.read_csv(pems_path)
    print(f"Loaded existing PeMS data: {len(df_pems)} rows.")

# Derive target latency delay per km = (1 / speed_mph - 1 / free_flow_speed) * 3600
free_flow_speed = 70.0
observed_delay_s_km = np.maximum(0.0, (1.0 / np.maximum(10.0, df_pems["avg_speed_mph"]) - 1.0 / free_flow_speed) * 3600.0)

lanes_feat = (df_pems["lanes"] / 4.0).values.astype(np.float32)
vc_ratio_feat = (df_pems["total_flow_veh_5min"] * 12.0 / (df_pems["lanes"] * 1800.0)).values.astype(np.float32)
occ_feat = (df_pems["avg_occupancy_pct"] / 100.0).values.astype(np.float32)
speed_norm = (df_pems["avg_speed_mph"] / 70.0).values.astype(np.float32)

X_pems = torch.tensor(np.stack([lanes_feat, vc_ratio_feat, occ_feat, speed_norm], axis=-1), dtype=torch.float32)
y_delay = torch.tensor(observed_delay_s_km.values.astype(np.float32), dtype=torch.float32)

# 2. Define Monotonic BPR Edge Cost Model
class MonotonicHazardLinear(nn.Linear):
    """Linear layer where weights are forced to be non-negative via exponentiation."""
    def forward(self, x):
        return F.linear(x, torch.exp(self.weight), self.bias)

class DynamicEdgeCostMLP(nn.Module):
    def __init__(self, road_feat_dim=4, tau_critical=0.75):
        super().__init__()
        self.tau_critical = tau_critical
        
        # Subnetwork 1: Baseline traffic latency
        self.traffic_subnet = nn.Sequential(
            nn.Linear(road_feat_dim, 16),
            nn.SiLU(),
            nn.Linear(16, 8),
            nn.SiLU(),
            nn.Linear(8, 1),
            nn.Softplus()
        )
        
        # Subnetwork 2: Hazard subnet with strictly positive exponential weights
        self.hazard_subnet = nn.Sequential(
            MonotonicHazardLinear(2, 4, bias=False),
            nn.ReLU(),
            MonotonicHazardLinear(4, 1, bias=False)
        )
        with torch.no_grad():
            self.hazard_subnet[0].weight.fill_(math.log(500.0))
            self.hazard_subnet[2].weight.fill_(math.log(2.0))

    def forward(self, base_time, road_features, burn_prob, pm25_val):
        traffic_delay = self.traffic_subnet(road_features).squeeze(-1)
        
        # Hazard inputs: [burn_prob, normalized smoke excess]
        smoke_excess = torch.relu(pm25_val - 35.0) / 100.0
        hazard_in = torch.stack([burn_prob, smoke_excess], dim=-1)
        hazard_delay = self.hazard_subnet(hazard_in).squeeze(-1)
        
        w_e = base_time + traffic_delay + hazard_delay
        
        # Hard structural closure threshold
        closed_mask = burn_prob >= self.tau_critical
        w_e = torch.where(closed_mask, torch.tensor(1e9, device=w_e.device), w_e)
        return w_e

# Train traffic delay subnet on PeMS
mlp_model = DynamicEdgeCostMLP(road_feat_dim=4)
opt_mlp = torch.optim.Adam(mlp_model.traffic_subnet.parameters(), lr=0.01, weight_decay=1e-4)

for _ in range(80):
    opt_mlp.zero_grad()
    pred_delay = mlp_model.traffic_subnet(X_pems).squeeze(-1)
    loss_pems = F.mse_loss(pred_delay, y_delay)
    loss_pems.backward()
    opt_mlp.step()

print(f"Trained DynamicEdgeCostMLP traffic subnet: final PeMS MSE = {loss_pems.item():.4f}")

# 3. Load Road Edges & Fuse with GNN Hazard Tensors
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_hazard = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet"))
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))

gdf_h3_poly = gpd.GeoDataFrame(
    df_hazard,
    geometry=gpd.GeoSeries.from_wkt(df_static["geometry_wkt"]),
    crs="EPSG:4326"
)

# Spatial STRtree lookup for edge risk assignment
strtree = STRtree(gdf_h3_poly.geometry.values)
edge_burn_t0, edge_burn_t3, edge_burn_t6, edge_burn_t12 = [], [], [], []

for _, edge in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([edge["geometry_wkt"]]).iloc[0]
    matched_idx = strtree.query(line_geom)
    
    if len(matched_idx) > 0:
        matched_cells = gdf_h3_poly.iloc[matched_idx]
        worst_cell = matched_cells.sort_values("burn_prob_12h", ascending=False).iloc[0]
    else:
        nearest_idx = strtree.nearest(line_geom.interpolate(0.5, normalized=True))
        worst_cell = gdf_h3_poly.iloc[nearest_idx]
        
    edge_burn_t0.append(worst_cell["burn_prob_0h"])
    edge_burn_t3.append(worst_cell["burn_prob_3h"])
    edge_burn_t6.append(worst_cell["burn_prob_6h"])
    edge_burn_t12.append(worst_cell["burn_prob_12h"])

df_edges["gnn_burn_prob_t0"] = np.array(edge_burn_t0, dtype=np.float32)
df_edges["gnn_burn_prob_t3"] = np.array(edge_burn_t3, dtype=np.float32)
df_edges["gnn_burn_prob_t6"] = np.array(edge_burn_t6, dtype=np.float32)
df_edges["gnn_burn_prob_t12"] = np.array(edge_burn_t12, dtype=np.float32)

# 4. Evaluate Dynamic Edge Costs W_e(t)
base_time_t = torch.tensor(df_edges["base_time_s"].values, dtype=torch.float32)
lanes_t = torch.tensor((df_edges["lanes"] / 4.0).values, dtype=torch.float32)
cap_t = torch.tensor((df_edges["capacity_veh_hr"] / 4000.0).values, dtype=torch.float32)
vc_t = torch.zeros(len(df_edges), dtype=torch.float32)  # Free-flow departure baseline
speed_t = torch.tensor((df_edges["speed_kmh"] / 110.0).values, dtype=torch.float32)

road_feat_tensor = torch.stack([lanes_t, cap_t, vc_t, speed_t], dim=-1)
pm25_tensor = torch.tensor(df_edges["pm25_ugm3"].values, dtype=torch.float32)

with torch.no_grad():
    w_t0 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t0"].values), pm25_tensor).numpy()
    w_t3 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t3"].values), pm25_tensor).numpy()
    w_t6 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t6"].values), pm25_tensor).numpy()
    w_t12 = mlp_model(base_time_t, road_feat_tensor, torch.tensor(df_edges["gnn_burn_prob_t12"].values), pm25_tensor).numpy()

df_edges["weight_t0_s"] = w_t0
df_edges["weight_t3_s"] = w_t3
df_edges["weight_t6_s"] = w_t6
df_edges["weight_t12_s"] = w_t12

# Closures summary
c0 = (w_t0 >= 1e8).sum()
c3 = (w_t3 >= 1e8).sum()
c6 = (w_t6 >= 1e8).sum()
c12 = (w_t12 >= 1e8).sum()

print("\n--- Monotonic Dynamic Road Impedance & Closures Summary ---")
print(f"Road Closures @ T=0h:  {c0} / {len(df_edges)}")
print(f"Road Closures @ T=3h:  {c3} / {len(df_edges)}")
print(f"Road Closures @ T=6h:  {c6} / {len(df_edges)}")
print(f"Road Closures @ T=12h: {c12} / {len(df_edges)}")

# 5. Save Model and Updated Graph
torch.save(mlp_model.state_dict(), os.path.join(models_dir, "monotonic_edge_cost_mlp.pt"))
df_edges.to_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"), compression="snappy")

# Build NetworkX graph
import networkx as nx
graph = nx.DiGraph()
for _, n in df_nodes.iterrows():
    graph.add_node(n["node_id"], name=n["name"], lon=n["lon"], lat=n["lat"])
for _, e in df_edges.iterrows():
    graph.add_edge(
        e["u"], e["v"],
        edge_id=e["edge_id"],
        length_m=e["length_m"],
        base_time_s=e["base_time_s"],
        capacity=e["capacity_veh_hr"],
        weights=[e["weight_t0_s"], e["weight_t3_s"], e["weight_t6_s"], e["weight_t12_s"]]
    )

with open(os.path.join(PROCESSED_DATA_DIR, "road_graph.pkl"), "wb") as f:
    pickle.dump(graph, f)

print(f"\nSaved updated monotonic road network: {os.path.join(PROCESSED_DATA_DIR, 'road_edges.parquet')}")

Generating Caltrans PeMS Butte County station baseline...
Saved 120 calibration rows to C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\raw\pems\pems_station_5min.csv
Trained DynamicEdgeCostMLP traffic subnet: final PeMS MSE = 149.4076

--- Monotonic Dynamic Road Impedance & Closures Summary ---
Road Closures @ T=0h:  18 / 34
Road Closures @ T=3h:  18 / 34
Road Closures @ T=6h:  30 / 34
Road Closures @ T=12h: 34 / 34

Saved updated monotonic road network: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\road_edges.parquet
